# Data preparation: dataset audit, cleaning and all-layer MERT features

**What this notebook does (in two parts):**

| Part | What happens | Typical time on a Colab T4 GPU |
|---|---|---|
| **A. Audit and cleaning** | Copies the raw datasets from Google Drive to the fast local Colab disk, unpacks them, reads every rating file, checks every audio file, removes unusable or duplicated recordings with an explicit reason, and saves a clean **catalog** | ~20–40 min (mostly copying Memo2496, 11.7 GB) |
| **B. Feature extraction** | Runs every catalog recording through the frozen MERT-v1-95M model and saves the output of **all 13 layers** for every 5-second window | ~1–3 h (estimate; the notebook prints the real speed and remaining time) |

**Datasets and roles** (fixed in the research log, Section 4):

| Dataset | Domain | Role |
|---|---|---|
| DEAM + EmoMusic | MediaEval | training/test domain (rotation) |
| PMEmo | PMEmo | training/test domain (rotation) |
| Memo2496 | Memo2496 | training/test domain (rotation) |
| Soundtracks (Set 2) | Soundtracks | **locked final test** — audio is processed, ratings are never read here |

**How to run:** Runtime → Change runtime type → **GPU**. Then Runtime → Run all.
If the installation cell asks for a restart, restart once and Run all again.
If Colab disconnects, simply Run all again: finished work is detected and skipped.

**What to send back at the end:** the file `data_preparation_review.zip` (created by the last cell).

## Step 1 — Settings (the only cell you may need to edit)

* `DATASETS_DIR`: the Google Drive folder that contains one sub-folder per dataset.
  Each sub-folder may contain the original downloaded archives (`.zip`, `.tar.gz`) and/or already-unpacked files; both work.
* `OUTPUT_DIR`: a **new** folder for all results of the new pipeline.

Everything below the line "fixed protocol settings" defines the experiment. Do not change it after extraction has started; the notebook refuses to mix features made with different settings.

In [6]:
from pathlib import Path

# ---------- paths: edit if your Drive layout is different ----------
DATASETS_DIR = Path('/content/drive/MyDrive/Thesis - Music Emotion Recognition/Datasets')
OUTPUT_DIR = Path('/content/drive/MyDrive/Thesis - Music Emotion Recognition/MER_DG_v2')
DATASET_FOLDERS = {            # dataset name -> sub-folder name inside DATASETS_DIR
    'DEAM': 'DEAM', 'EmoMusic': 'EmoMusic', 'PMEmo': 'PMEmo',
    'Memo2496': 'Memo2496', 'Soundtracks': 'Soundtracks',
}

# ---------- fixed protocol settings ----------
LOCAL_RAW = Path('/content/raw')   # fast local disk; rebuilt in every new Colab session
DOMAIN_OF = {'DEAM': 'MediaEval', 'EmoMusic': 'MediaEval', 'PMEmo': 'PMEmo',
             'Memo2496': 'Memo2496', 'Soundtracks': 'Soundtracks'}
# When the same recording or artist occurs in two domains, it is kept only in the
# earliest domain of this list (smaller / more precious domains first).
DOMAIN_PRIORITY = ['Soundtracks', 'PMEmo', 'MediaEval', 'Memo2496']
DATASET_PRIORITY = ['Soundtracks', 'PMEmo', 'EmoMusic', 'DEAM', 'Memo2496']
MIN_DURATION_S = 10.0                       # shorter recordings are excluded
MEMO_LABEL_WINDOW_S = (15, 75)              # Memo2496 song label = mean rating in this window
MEMO_SHORT_TRACK_S = 30                     # tracks shorter than this use the whole track
SOUNDTRACKS_SET = 'set2'
SOUNDTRACKS_EXCLUDE = {'095': 'identical audio to track 010 (stated in the dataset readme)'}
GENERIC_ARTISTS = {'', 'unknown', 'various', 'various artists', 'n a', 'na', 'none', 'nan', 'anonymous'}

MODEL_ID = 'm-a-p/MERT-v1-95M'
SAMPLE_RATE = 24000
WINDOW_S = 5.0                 # non-overlapping analysis windows
MIN_LAST_WINDOW_S = 1.0        # a shorter final remainder is merged into the previous window
BATCH_WINDOWS = 16             # windows per GPU batch
PREFETCH_TRACKS = 16           # recordings decoded in parallel while the GPU works

PROTOCOL = dict(
    version='data-preparation-v1', domain_of=DOMAIN_OF, domain_priority=DOMAIN_PRIORITY,
    dataset_priority=DATASET_PRIORITY, min_duration_s=MIN_DURATION_S,
    memo_label_window_s=list(MEMO_LABEL_WINDOW_S), memo_short_track_s=MEMO_SHORT_TRACK_S,
    soundtracks_set=SOUNDTRACKS_SET, soundtracks_exclude=SOUNDTRACKS_EXCLUDE,
    generic_artists=sorted(GENERIC_ARTISTS),
    label_maps={'DEAM': '(x-1)/8 from 1..9', 'EmoMusic': '(x-1)/8 from 1..9',
                'PMEmo': 'as published, 0..1', 'Memo2496': '(x+1)/2 from -1..1',
                'Soundtracks': 'locked; not read in this notebook'},
)
print('Settings loaded. Datasets folder:', DATASETS_DIR)
print('Outputs will be written to:', OUTPUT_DIR)

Settings loaded. Datasets folder: /content/drive/MyDrive/Thesis - Music Emotion Recognition/Datasets
Outputs will be written to: /content/drive/MyDrive/Thesis - Music Emotion Recognition/MER_DG_v2


## Step 2 — Install fixed library versions

MERT needs specific versions of `transformers` and `nnAudio`. Fixed versions make the extraction reproducible.
If something is installed, the cell asks for **one runtime restart** (Runtime → Restart session), after which you press Run all again.

In [7]:
import sys, subprocess, importlib.metadata
if not ((3, 10) <= sys.version_info[:2] <= (3, 12)):
    raise RuntimeError('Please use a Colab runtime with Python 3.10–3.12 (Runtime → Change runtime type → runtime version).')
PINS = ['transformers==4.38.2', 'huggingface-hub==0.23.5', 'nnAudio==0.3.3', 'soundfile==0.12.1',
        'numpy==1.26.4', 'pandas==2.2.2', 'scipy==1.13.1', 'scikit-learn==1.5.1']
missing = []
for pin in PINS:
    name, version = pin.split('==')
    try:
        if importlib.metadata.version(name) != version:
            missing.append(pin)
    except importlib.metadata.PackageNotFoundError:
        missing.append(pin)
if missing:
    print('Installing:', ', '.join(missing), flush=True)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *PINS])
    raise RuntimeError('✅ Installed. Now do Runtime → Restart session, then Run all again. '
                       '(This message is expected once; it stops Run all on purpose.)')
print('✅ Library versions are correct.')

✅ Library versions are correct.


## Step 3 — Connect Google Drive and check the dataset folders

The cell mounts Drive, checks that every dataset folder exists, and lists what is inside (names and sizes only).
If a folder is missing, it searches your Drive for folders with the right name and prints suggestions for `DATASETS_DIR`.

In [8]:
from google.colab import drive
drive.mount('/content/drive')
import os

def human_size(n):
    for unit in ('B', 'KB', 'MB', 'GB'):
        if n < 1024:
            return f'{n:.1f} {unit}'
        n /= 1024
    return f'{n:.1f} TB'

def find_dirs(root, names, max_depth=4):
    """Breadth-limited search for folders with the given names (used only for suggestions)."""
    found, frontier = [], [(Path(root), 0)]
    wanted = {n.lower() for n in names}
    while frontier:
        folder, depth = frontier.pop()
        try:
            children = [c for c in folder.iterdir() if c.is_dir()]
        except OSError:
            continue
        for child in children:
            if child.name.lower() in wanted:
                found.append(child)
            if depth + 1 < max_depth:
                frontier.append((child, depth + 1))
    return found

problems = []
for name, sub in DATASET_FOLDERS.items():
    folder = DATASETS_DIR / sub
    if not folder.is_dir():
        problems.append(name)
        print(f'❌ {name}: folder not found: {folder}')
        continue
    print(f'✅ {name}: {folder}')
    for item in sorted(folder.iterdir()):
        size = item.stat().st_size if item.is_file() else None
        print(f'      {"📄" if item.is_file() else "📁"} {item.name}' + (f'  ({human_size(size)})' if size else ''))
if problems:
    print('\nSearching your Drive for folders with these names (this can take a minute)...')
    for hit in find_dirs('/content/drive/MyDrive', [DATASET_FOLDERS[p] for p in problems]):
        print('   found:', hit)
    raise FileNotFoundError('Fix DATASETS_DIR / DATASET_FOLDERS in Step 1 using the suggestions above, then Run all.')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('\n✅ All dataset folders found. Output folder ready:', OUTPUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ DEAM: /content/drive/MyDrive/Thesis - Music Emotion Recognition/Datasets/DEAM
      📄 DEAM_Annotations.zip  (4.5 MB)
      📄 DEAM_audio.zip  (1.3 GB)
      📄 metadata.zip  (336.7 KB)
✅ EmoMusic: /content/drive/MyDrive/Thesis - Music Emotion Recognition/Datasets/EmoMusic
      📄 annotations.tar.gz  (1.3 MB)
      📄 clips_45sec.tar.gz  (458.7 MB)
✅ PMEmo: /content/drive/MyDrive/Thesis - Music Emotion Recognition/Datasets/PMEmo
      📄 PMEmo2019.zip  (648.2 MB)
      📄 README.txt  (1.5 KB)
✅ Memo2496: /content/drive/MyDrive/Thesis - Music Emotion Recognition/Datasets/Memo2496
      📄 25827034.zip  (27.9 MB)
      📄 MusicRawData.zip  (11.5 GB)
✅ Soundtracks: /content/drive/MyDrive/Thesis - Music Emotion Recognition/Datasets/Soundtracks
      📄 p6vkg-osfstorage-archive.zip  (203.9 MB)

✅ All dataset folders found. Output folder ready: /content/drive/MyDrive/Thes

## Step 4 — Helper functions

Small tools used everywhere below: timestamped messages, a progress reporter that prints the percentage done, the elapsed time and the **estimated remaining time**, safe file writing, checksums, and text normalization for comparing artist and song names.

In [9]:
import csv, datetime, hashlib, json, math, re, shutil, tarfile, time, unicodedata, zipfile
import numpy as np
import pandas as pd

def log(message):
    print(f'[{datetime.datetime.now():%H:%M:%S}] {message}', flush=True)

def fmt_seconds(s):
    if s is None or not math.isfinite(s):
        return '?'
    s = int(s)
    return f'{s // 3600}h {s % 3600 // 60:02d}m' if s >= 3600 else f'{s // 60}m {s % 60:02d}s'

class Progress:
    """Prints done/total, percent, elapsed time and ETA at most every `every_s` seconds."""
    def __init__(self, total, label, every_s=30, unit='items'):
        self.total, self.label, self.every_s, self.unit = max(int(total), 1), label, every_s, unit
        self.done, self.start, self.last = 0, time.monotonic(), -1e9
        log(f'▶ {label}: starting' + ('' if unit == 'bytes' else f' ({total} {unit})'))
    def step(self, n=1, note=''):
        self.done += n
        now = time.monotonic()
        if self.done >= self.total or now - self.last >= self.every_s:
            elapsed = now - self.start
            rate = self.done / elapsed if elapsed > 0 else 0
            eta = (self.total - self.done) / rate if rate > 0 else None
            amount = (f'{human_size(self.done)} / {human_size(self.total)}' if self.unit == 'bytes'
                      else f'{self.done}/{self.total}')
            log(f'  {self.label}: {amount} ({100 * self.done / self.total:.1f}%) | '
                f'elapsed {fmt_seconds(elapsed)} | remaining ≈ {fmt_seconds(eta)} {note}')
            self.last = now
    def finish(self):
        log(f'✔ {self.label}: finished in {fmt_seconds(time.monotonic() - self.start)}')

def sha256_file(path, chunk=8 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

def sha256_json(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=False).encode('utf-8')).hexdigest()

def atomic_write_text(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(text, encoding='utf-8')
    os.replace(tmp, path)

def _plain(value):
    """Lets json write numpy numbers and paths."""
    return value.item() if hasattr(value, 'item') else str(value)

def atomic_json(path, obj):
    atomic_write_text(path, json.dumps(obj, indent=2, ensure_ascii=False, default=_plain))

def is_junk(path):
    path = Path(path)
    return '__MACOSX' in path.parts or path.name.startswith('._') or path.name == '.DS_Store'

def norm_text(value):
    """Lower-case, accent-free, punctuation-free text; drops (...) [...] and 'feat.' parts."""
    s = unicodedata.normalize('NFKD', str(value or ''))
    s = ''.join(c for c in s if not unicodedata.combining(c)).casefold()
    s = re.sub(r'\(.*?\)|\[.*?\]', ' ', s)
    s = re.split(r'\b(?:feat|ft|featuring)\b', s)[0]
    s = re.sub(r'[\W_]+', ' ', s)
    return ' '.join(s.split())

def canonical_id(value):
    value = str(value).strip()
    return str(int(float(value))) if re.fullmatch(r'\d+(?:\.0+)?', value) else value

PROTOCOL_HASH = sha256_json(PROTOCOL)
log(f'Helpers ready. Protocol fingerprint: {PROTOCOL_HASH[:12]}')

[08:57:10] Helpers ready. Protocol fingerprint: 8c36a5628879


# Part A — Dataset audit and cleaning

## Step A1 — Copy the raw datasets to the local Colab disk and unpack them

Reading thousands of audio files directly from Google Drive is slow, so each dataset folder is copied once to the local disk (`/content/raw`) and every archive inside it is unpacked (also archives inside archives, as in Soundtracks).
Google Drive copies a few large files quickly but thousands of small files very slowly, so **when a dataset folder contains the original archives (`.zip`, `.tar.gz`), only the archives are copied** and loose, already-unpacked files are skipped. A folder without archives is copied completely.
macOS helper files (`__MACOSX`, `._*`) are ignored. The local copy of an archive is deleted after unpacking to save disk space; **nothing on your Drive is changed or deleted**.
In a new Colab session this step runs again (the local disk is empty in a new session).

In [10]:
ARCHIVE_ENDINGS = ('.zip', '.tar.gz', '.tgz', '.tar')

def copy_with_progress(src, dst, progress):
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.part')
    with open(src, 'rb') as fin, open(tmp, 'wb') as fout:
        for block in iter(lambda: fin.read(32 * 1024 * 1024), b''):
            fout.write(block)
            progress.step(len(block))
    os.replace(tmp, dst)

def extract_archive(archive, dest):
    dest.mkdir(parents=True, exist_ok=True)
    name = archive.name.lower()
    if name.endswith('.zip'):
        with zipfile.ZipFile(archive) as z:
            members = [m for m in z.infolist() if not is_junk(m.filename)]
            for m in members:
                z.extract(m, dest)
            return len(members)
    with tarfile.open(archive, 'r:*') as t:
        members = [m for m in t.getmembers() if not is_junk(m.name) and (m.isfile() or m.isdir())]
        try:
            t.extractall(dest, members=members, filter='data')
        except TypeError:  # Python without the tar 'filter' argument
            t.extractall(dest, members=members)
        return len(members)

def materialize(dataset):
    """Copy one Drive dataset folder to LOCAL_RAW and unpack all archives (recursively)."""
    src_root, dst_root = DATASETS_DIR / DATASET_FOLDERS[dataset], LOCAL_RAW / dataset
    files = sorted(p for p in src_root.rglob('*') if p.is_file() and not is_junk(p))
    archives = [p for p in files if p.name.lower().endswith(ARCHIVE_ENDINGS)]
    if archives:
        # Google Drive is very slow for thousands of small files but fast for a few large ones.
        # When the original archives are present, copy only them; loose (already unpacked) files are skipped.
        skipped = len(files) - len(archives)
        files = archives
        if skipped:
            log(f'{dataset}: copying only the {len(archives)} archive(s); {skipped} loose files on Drive are skipped '
                f'(the same content is inside the archives).')
    listing = [(str(p.relative_to(src_root)), p.stat().st_size) for p in files]
    marker = dst_root / '.ready.json'
    if marker.is_file() and json.loads(marker.read_text())['listing'] == listing:
        log(f'{dataset}: already copied and unpacked in this session, skipping.')
        return
    if dst_root.exists():
        shutil.rmtree(dst_root)
    total = sum(size for _, size in listing)
    progress = Progress(total, f'{dataset}: copying {len(files)} files ({human_size(total)}) from Drive',
                        every_s=20, unit='bytes')
    for p in files:
        copy_with_progress(p, dst_root / p.relative_to(src_root), progress)
    progress.finish()
    done = set()
    while True:
        archives = [p for p in dst_root.rglob('*') if p.is_file() and not is_junk(p)
                    and p.name.lower().endswith(ARCHIVE_ENDINGS) and p not in done]
        if not archives:
            break
        for archive in archives:
            log(f'{dataset}: unpacking {archive.name} ({human_size(archive.stat().st_size)}) ...')
            n = extract_archive(archive, archive.with_name(archive.name + '__unpacked'))
            log(f'{dataset}: unpacked {n} entries from {archive.name}')
            done.add(archive)
            archive.unlink()  # local copy only; the Drive original is untouched
    atomic_json(marker, {'listing': listing})

LOCAL_RAW.mkdir(parents=True, exist_ok=True)
free = shutil.disk_usage('/content').free
log(f'Free local disk: {human_size(free)}')
for dataset in DATASET_FOLDERS:
    materialize(dataset)
log('✅ Step A1 done: all datasets are on the local disk.')

[08:57:10] Free local disk: 194.0 GB
[08:57:10] ▶ DEAM: copying 3 files (1.3 GB) from Drive: starting
[08:57:10]   DEAM: copying 3 files (1.3 GB) from Drive: 4.5 MB / 1.3 GB (0.4%) | elapsed 0m 00s | remaining ≈ 3m 06s 
[08:57:31]   DEAM: copying 3 files (1.3 GB) from Drive: 740.5 MB / 1.3 GB (57.6%) | elapsed 0m 21s | remaining ≈ 0m 15s 
[08:57:48]   DEAM: copying 3 files (1.3 GB) from Drive: 1.3 GB / 1.3 GB (100.0%) | elapsed 0m 38s | remaining ≈ 0m 00s 
[08:57:48] ✔ DEAM: copying 3 files (1.3 GB) from Drive: finished in 0m 38s
[08:57:48] DEAM: unpacking DEAM_audio.zip (1.3 GB) ...
[08:57:57] DEAM: unpacked 1803 entries from DEAM_audio.zip
[08:57:57] DEAM: unpacking metadata.zip (336.7 KB) ...
[08:57:57] DEAM: unpacked 4 entries from metadata.zip
[08:57:57] DEAM: unpacking DEAM_Annotations.zip (4.5 MB) ...
[08:57:58] DEAM: unpacked 3619 entries from DEAM_Annotations.zip
[08:57:58] ▶ EmoMusic: copying 2 files (460.1 MB) from Drive: starting
[08:57:59]   EmoMusic: copying 2 files (460.

## Step A2 — Find the rating files and audio folders

For each dataset the cell looks for the files by their **original names** from the official releases and prints where it found them.
If a file is missing or ambiguous, the error message says exactly which file and where it looked.

| Dataset | Ratings | Metadata (artist, title) | Audio |
|---|---|---|---|
| DEAM | `static_annotations_averaged_songs_1_2000.csv`, `..._2000_2058.csv` | `metadata_2014.csv`, `metadata_2015.csv` | `MEMD_audio/*.mp3` |
| EmoMusic | `static_annotations.csv` | `songs_info.csv` | `clips_45seconds/*.mp3` |
| PMEmo | `static_annotations.csv` | `metadata.csv` | `chorus/*.mp3` |
| Memo2496 | `valence_all_average.csv`, `arousal_all_average.csv` (per second) | `songs_info_all.csv` | UUID-named `*.mp3` |
| Soundtracks | `mean_ratings_set2.csv` (**only its checksum is recorded**) | `set2_tracklist.csv` | `set2/*.mp3` |

In [11]:
def find_one(dataset, filename):
    root = LOCAL_RAW / dataset
    hits = sorted(p for p in root.rglob(filename) if p.is_file() and not is_junk(p))
    if not hits:
        raise FileNotFoundError(f'{dataset}: could not find "{filename}" under {DATASETS_DIR / DATASET_FOLDERS[dataset]}. '
                                f'Upload the original dataset archive or file to that folder.')
    if len({sha256_file(h) for h in hits}) > 1:
        raise ValueError(f'{dataset}: several DIFFERENT files named "{filename}": {hits}. Keep only the original one on Drive.')
    return hits[0]

UUID = r'[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}'

def audio_index(dataset, folder_name=None, stem_pattern=r'\d+'):
    """Map file stem -> audio path. Identical copies in several places are tolerated."""
    root = LOCAL_RAW / dataset
    candidates = [p for p in root.rglob('*') if p.is_file() and not is_junk(p)
                  and p.suffix.lower() in ('.mp3', '.wav', '.flac', '.ogg', '.m4a')
                  and re.fullmatch(stem_pattern, p.stem, flags=re.I)]
    if folder_name:
        inside = [p for p in candidates if p.parent.name.lower() == folder_name.lower()]
        if inside:
            candidates = inside
        else:
            log(f'⚠ {dataset}: no folder named "{folder_name}"; using all matching audio files.')
    index = {}
    for p in sorted(candidates):
        index.setdefault(p.stem.lower() if dataset == 'Memo2496' else canonical_id(p.stem), []).append(p)
    resolved = {}
    for stem, paths in index.items():
        if len({p.stat().st_size for p in paths}) > 1:
            raise ValueError(f'{dataset}: different audio files share the name {stem}: {paths}')
        resolved[stem] = paths[0]
    return resolved

PATHS = {
    'DEAM': dict(static_1=find_one('DEAM', 'static_annotations_averaged_songs_1_2000.csv'),
                 static_2=find_one('DEAM', 'static_annotations_averaged_songs_2000_2058.csv'),
                 meta_2014=find_one('DEAM', 'metadata_2014.csv'),
                 meta_2015=find_one('DEAM', 'metadata_2015.csv')),
    'EmoMusic': dict(static=find_one('EmoMusic', 'static_annotations.csv'),
                     info=find_one('EmoMusic', 'songs_info.csv')),
    'PMEmo': dict(static=find_one('PMEmo', 'static_annotations.csv'),
                  meta=find_one('PMEmo', 'metadata.csv')),
    'Memo2496': dict(valence=find_one('Memo2496', 'valence_all_average.csv'),
                     arousal=find_one('Memo2496', 'arousal_all_average.csv'),
                     info=find_one('Memo2496', 'songs_info_all.csv')),
    'Soundtracks': dict(ratings_locked=find_one('Soundtracks', f'mean_ratings_{SOUNDTRACKS_SET}.csv'),
                        tracklist=find_one('Soundtracks', f'{SOUNDTRACKS_SET}_tracklist.csv')),
}
AUDIO = {
    'DEAM': audio_index('DEAM', 'MEMD_audio'),
    'EmoMusic': audio_index('EmoMusic', 'clips_45seconds'),
    'PMEmo': audio_index('PMEmo', 'chorus'),
    'Memo2496': audio_index('Memo2496', None, UUID),
    'Soundtracks': audio_index('Soundtracks', SOUNDTRACKS_SET, r'\d{3}'),
}
for dataset, files in PATHS.items():
    print(f'📂 {dataset}: {len(AUDIO[dataset])} audio files')
    for key, path in files.items():
        print(f'     {key:15s} {path.relative_to(LOCAL_RAW)}')
log('✅ Step A2 done.')

📂 DEAM: 1802 audio files
     static_1        DEAM/DEAM_Annotations.zip__unpacked/annotations/annotations averaged per song/song_level/static_annotations_averaged_songs_1_2000.csv
     static_2        DEAM/DEAM_Annotations.zip__unpacked/annotations/annotations averaged per song/song_level/static_annotations_averaged_songs_2000_2058.csv
     meta_2014       DEAM/metadata.zip__unpacked/metadata/metadata_2014.csv
     meta_2015       DEAM/metadata.zip__unpacked/metadata/metadata_2015.csv
📂 EmoMusic: 1000 audio files
     static          EmoMusic/annotations.tar.gz__unpacked/static_annotations.csv
     info            EmoMusic/annotations.tar.gz__unpacked/songs_info.csv
📂 PMEmo: 794 audio files
     static          PMEmo/PMEmo2019.zip__unpacked/PMEmo2019/annotations/static_annotations.csv
     meta            PMEmo/PMEmo2019.zip__unpacked/PMEmo2019/metadata.csv
📂 Memo2496: 2496 audio files
     valence         Memo2496/25827034.zip__unpacked/Annotations/valence_all_average.csv
     arousal

## Step A3 — Read ratings and metadata, and convert every label to the 0–1 scale

Each dataset uses a different rating scale. Every label is converted with a **fixed formula that comes from the definition of the scale**, never from the data, so no information about any test set is used:

| Dataset | Original scale | Conversion |
|---|---|---|
| DEAM, EmoMusic | 1 to 9 | (x − 1) / 8 |
| PMEmo | 0 to 1 | unchanged |
| Memo2496 | −1 to 1, one value per second | mean over 15–75 s (whole track if shorter than 30 s), then (x + 1) / 2 |
| Soundtracks | 1 to 9 | **not read here** (locked final test) |

Rules applied while reading (each removed recording gets a written reason):
* DEAM songs with ID ≤ 1000 are the 2013 MediaEval set, which **is** EmoMusic. They are taken from EmoMusic, not twice.
* A rating without audio, or audio without a rating, cannot be used.
* Soundtracks track 095 is the same audio as track 010 (dataset readme).

In [12]:
def read_table(path):
    """Robust CSV reader: strips spaces/tabs, ignores extra columns, tries common encodings."""
    for encoding in ('utf-8-sig', 'cp1252', 'latin-1'):
        try:
            with open(path, newline='', encoding=encoding) as f:
                rows = list(csv.reader(f))
            break
        except UnicodeDecodeError:
            continue
    header = [h.strip() for h in rows[0]]
    records = []
    for row in rows[1:]:
        if not any(cell.strip() for cell in row):
            continue
        row = [cell.strip() for cell in row] + [''] * max(0, len(header) - len(row))
        records.append(dict(zip(header, row[:len(header)])))
    return header, records

def col(record, *names):
    lookup = {k.casefold(): v for k, v in record.items()}
    for name in names:
        if name.casefold() in lookup:
            return lookup[name.casefold()]
    raise KeyError(f'None of the columns {names} found; available: {list(record)}')

CANDIDATES, EXCLUDED, MEMO_SERIES = [], [], {}

def exclude(dataset, track_id, reason, detail=''):
    EXCLUDED.append(dict(uid=f'{dataset}:{track_id}', dataset=dataset, track_id=str(track_id),
                         reason=reason, detail=detail))

def add_candidate(dataset, track_id, audio, artist='', title='', valence=None, arousal=None,
                  label_note='', group_hint=None):
    artist_key = norm_text(artist)
    if group_hint:
        group = group_hint
    elif artist_key not in GENERIC_ARTISTS:
        group = f'artist:{artist_key}'
    else:
        group = f'track:{dataset}:{track_id}'
    CANDIDATES.append(dict(uid=f'{dataset}:{track_id}', dataset=dataset, domain=DOMAIN_OF[dataset],
                           track_id=str(track_id), audio_relpath=str(audio.relative_to(LOCAL_RAW)),
                           artist=artist, title=title, artist_key=artist_key, title_key=norm_text(title),
                           group_id=group, valence=valence, arousal=arousal, label_note=label_note,
                           label_locked=dataset == 'Soundtracks'))

def check_range(values, low, high, where):
    for v in values:
        if not (math.isfinite(v) and low <= v <= high):
            raise ValueError(f'{where}: rating {v} outside the documented range {low}..{high}')

def attach_audio(dataset, labelled_ids):
    """Record audio files that have no rating."""
    for stem in sorted(set(AUDIO[dataset]) - set(labelled_ids)):
        exclude(dataset, stem, 'audio without rating')

# ---------- DEAM (2014 set: IDs 1001-2000; 2015 full songs: IDs 2001-2058) ----------
_, meta14 = read_table(PATHS['DEAM']['meta_2014'])
_, meta15 = read_table(PATHS['DEAM']['meta_2015'])
deam_meta = {canonical_id(col(r, 'Id')): (col(r, 'Artist'), col(r, 'Track')) for r in meta14}
deam_meta.update({canonical_id(col(r, 'id')): (col(r, 'artist'), col(r, 'title')) for r in meta15})
deam_ids = []
for key in ('static_1', 'static_2'):
    _, rows = read_table(PATHS['DEAM'][key])
    for r in rows:
        tid = canonical_id(col(r, 'song_id'))
        deam_ids.append(tid)
        if int(tid) <= 1000:
            exclude('DEAM', tid, 'DEAM 2013 subset (identical to the EmoMusic catalogue; used via EmoMusic)')
            continue
        v, a = float(col(r, 'valence_mean')), float(col(r, 'arousal_mean'))
        check_range([v, a], 1, 9, f'DEAM {tid}')
        if tid not in AUDIO['DEAM']:
            exclude('DEAM', tid, 'rating without audio')
            continue
        artist, title = deam_meta.get(tid, ('', ''))
        note = 'full song (2015 set)' if int(tid) > 2000 else '45-s excerpt (2014 set)'
        add_candidate('DEAM', tid, AUDIO['DEAM'][tid], artist, title, (v - 1) / 8, (a - 1) / 8, note)
attach_audio('DEAM', deam_ids)

# ---------- EmoMusic ----------
_, info = read_table(PATHS['EmoMusic']['info'])
emo_meta = {canonical_id(col(r, 'song_id')): (col(r, 'Artist'), col(r, 'Song title')) for r in info}
_, rows = read_table(PATHS['EmoMusic']['static'])
emo_ids = []
for r in rows:
    tid = canonical_id(col(r, 'song_id'))
    emo_ids.append(tid)
    v, a = float(col(r, 'mean_valence')), float(col(r, 'mean_arousal'))
    check_range([v, a], 1, 9, f'EmoMusic {tid}')
    if tid not in AUDIO['EmoMusic']:
        exclude('EmoMusic', tid, 'rating without audio')
        continue
    artist, title = emo_meta.get(tid, ('', ''))
    add_candidate('EmoMusic', tid, AUDIO['EmoMusic'][tid], artist, title, (v - 1) / 8, (a - 1) / 8, '45-s excerpt')
attach_audio('EmoMusic', emo_ids)

# ---------- PMEmo ----------
_, meta = read_table(PATHS['PMEmo']['meta'])
pm_meta = {canonical_id(col(r, 'musicId')): (col(r, 'artist'), col(r, 'title'), Path(col(r, 'fileName')).stem) for r in meta}
_, rows = read_table(PATHS['PMEmo']['static'])
pm_ids = []
for r in rows:
    tid = canonical_id(col(r, 'musicId'))
    pm_ids.append(tid)
    v, a = float(col(r, 'Valence(mean)')), float(col(r, 'Arousal(mean)'))
    check_range([v, a], 0, 1, f'PMEmo {tid}')
    artist, title, stem = pm_meta.get(tid, ('', '', tid))
    stem = canonical_id(stem)
    if stem not in AUDIO['PMEmo']:
        exclude('PMEmo', tid, 'rating without audio')
        continue
    add_candidate('PMEmo', tid, AUDIO['PMEmo'][stem], artist, title, v, a, 'chorus excerpt')
attach_audio('PMEmo', pm_ids)

# ---------- Memo2496 (per-second ratings -> one song label) ----------
def read_series(path):
    with open(path, newline='', encoding='utf-8-sig') as f:
        reader = csv.reader(f)
        next(reader)
        return {canonical_id(row[0]): np.array([float(x) for x in row[1:] if x.strip() != ''], dtype=np.float64)
                for row in reader if row and row[0].strip()}
memo_v, memo_a = read_series(PATHS['Memo2496']['valence']), read_series(PATHS['Memo2496']['arousal'])
_, info = read_table(PATHS['Memo2496']['info'])
memo_ids = []
for r in info:
    tid = canonical_id(col(r, 'song_id'))
    stem = Path(col(r, 'file_name')).stem.lower()
    memo_ids.append(stem)
    if tid not in memo_v or tid not in memo_a:
        exclude('Memo2496', tid, 'no rating series')
        continue
    v, a = memo_v[tid], memo_a[tid]
    if len(v) != len(a) or not len(v):
        exclude('Memo2496', tid, 'valence/arousal series lengths differ or are empty')
        continue
    check_range(np.concatenate([v, a]), -1, 1, f'Memo2496 {tid}')
    start, end = MEMO_LABEL_WINDOW_S
    if len(v) < MEMO_SHORT_TRACK_S:
        start, end, note = 0, len(v), f'whole track ({len(v)} s rated; shorter than {MEMO_SHORT_TRACK_S} s)'
    else:
        end = min(end, len(v))
        note = f'mean of seconds {start}-{end}'
    if stem not in AUDIO['Memo2496']:
        exclude('Memo2496', tid, 'rating without audio')
        continue
    MEMO_SERIES[f'Memo2496:{tid}'] = np.stack([v, a])
    add_candidate('Memo2496', tid, AUDIO['Memo2496'][stem], '', col(r, 'Song title'),
                  (float(v[start:end].mean()) + 1) / 2, (float(a[start:end].mean()) + 1) / 2, note)
attach_audio('Memo2496', memo_ids)

# ---------- Soundtracks (ratings LOCKED: only their checksum is stored) ----------
_, tracks = read_table(PATHS['Soundtracks']['tracklist'])
st_album = {col(r, 'Nro').zfill(3): col(r, 'Album name') for r in tracks}
for stem in sorted(AUDIO['Soundtracks']):
    tid = stem.zfill(3)
    if tid in SOUNDTRACKS_EXCLUDE:
        exclude('Soundtracks', tid, 'duplicate excerpt', SOUNDTRACKS_EXCLUDE[tid])
        continue
    album = st_album.get(tid, '')
    add_candidate('Soundtracks', tid, AUDIO['Soundtracks'][stem], '', album, None, None,
                  'locked final test; ratings not read', group_hint=f'film:{norm_text(album) or tid}')
SOUNDTRACKS_LOCK = dict(ratings_file=str(PATHS['Soundtracks']['ratings_locked'].relative_to(LOCAL_RAW)),
                        ratings_sha256=sha256_file(PATHS['Soundtracks']['ratings_locked']),
                        note='Ratings were NOT opened in Notebook 16. The final test must verify this checksum.')

summary = pd.DataFrame(CANDIDATES).groupby('dataset').size().rename('candidates').to_frame()
summary['excluded so far'] = pd.DataFrame(EXCLUDED, columns=['uid', 'dataset', 'track_id', 'reason', 'detail']).groupby('dataset').size()
print(summary.fillna(0).astype(int))
log('✅ Step A3 done: labels read and converted.')

             candidates  excluded so far
dataset                                 
DEAM               1058              744
EmoMusic            744              256
Memo2496           2496                0
PMEmo               767               27
Soundtracks         109                1
[09:07:32] ✅ Step A3 done: labels read and converted.


## Step A4 — Check every audio file

For every candidate recording the cell measures the duration, computes a fingerprint (SHA-256 checksum of the file bytes, used to find identical files) and excludes recordings that cannot be opened or are shorter than 10 seconds.
The progress line shows how many files are done and the remaining time.

In [13]:
import soundfile as sf

def audio_duration(path):
    try:
        return float(sf.info(str(path)).duration)
    except Exception:
        out = subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'csv=p=0', str(path)],
                             capture_output=True, text=True)
        return float(out.stdout.strip())

checked = []
progress = Progress(len(CANDIDATES), 'Checking audio files', every_s=30)
for rec in CANDIDATES:
    path = LOCAL_RAW / rec['audio_relpath']
    try:
        size = path.stat().st_size
        if size == 0:
            raise ValueError('empty file')
        duration = audio_duration(path)
    except Exception as error:
        exclude(rec['dataset'], rec['track_id'], 'audio cannot be opened', str(error)[:200])
        progress.step()
        continue
    if duration < MIN_DURATION_S:
        exclude(rec['dataset'], rec['track_id'], f'shorter than {MIN_DURATION_S:.0f} s', f'{duration:.1f} s')
    else:
        checked.append(dict(rec, duration_s=round(duration, 3), audio_bytes=size, audio_sha256=sha256_file(path)))
    progress.step()
progress.finish()
CANDIDATES = checked
log(f'✅ Step A4 done: {len(CANDIDATES)} recordings passed the audio checks.')

[09:07:32] ▶ Checking audio files: starting (5174 items)
[09:07:32]   Checking audio files: 1/5174 (0.0%) | elapsed 0m 00s | remaining ≈ 5m 11s 
[09:08:02]   Checking audio files: 3701/5174 (71.5%) | elapsed 0m 30s | remaining ≈ 0m 11s 
[09:08:22]   Checking audio files: 5174/5174 (100.0%) | elapsed 0m 50s | remaining ≈ 0m 00s 
[09:08:22] ✔ Checking audio files: finished in 0m 50s
[09:08:22] ✅ Step A4 done: 5174 recordings passed the audio checks.


## Step A5 — Remove duplicates and cross-domain artist overlap

Every domain is used as an unseen test domain in one fold, so a recording (or an artist) shared by two domains would let the model "see" part of a test set during training. Three rules are applied in this order; the copy in the **higher-priority** domain is kept (priority: Soundtracks > PMEmo > MediaEval > Memo2496, i.e. smaller domains first):

1. **Identical audio files** (same checksum), anywhere.
2. **Same artist and same title** (after normalizing spelling), anywhere. Only DEAM, EmoMusic and PMEmo have artist names.
3. **Same artist in two different domains**: that artist's recordings are kept only in the higher-priority domain.

Near-identical audio that is not byte-identical (different encodings of the same song) cannot be detected from metadata alone; Step B5 reports such pairs from the MERT features for manual review.

In [14]:
def priority(rec):
    tid = rec['track_id']
    return (DOMAIN_PRIORITY.index(rec['domain']), DATASET_PRIORITY.index(rec['dataset']),
            (0, int(tid)) if tid.isdigit() else (1, tid))

def dedupe(records, key_fn, reason):
    groups = {}
    for rec in records:
        key = key_fn(rec)
        if key is not None:
            groups.setdefault(key, []).append(rec)
    removed = set()
    for members in groups.values():
        if len(members) > 1:
            members = sorted(members, key=priority)
            for rec in members[1:]:
                removed.add(rec['uid'])
                exclude(rec['dataset'], rec['track_id'], reason, f'kept {members[0]["uid"]}')
    log(f'{reason}: {len(removed)} recordings removed')
    return [r for r in records if r['uid'] not in removed]

records = dedupe(CANDIDATES, lambda r: r['audio_sha256'], 'identical audio file')
records = dedupe(records, lambda r: (r['artist_key'], r['title_key'])
                 if r['artist_key'] not in GENERIC_ARTISTS and r['title_key'] else None, 'same artist and title')

artist_domains = {}
for r in records:
    if r['artist_key'] not in GENERIC_ARTISTS:
        artist_domains.setdefault(r['artist_key'], set()).add(r['domain'])
shared = {a: min(ds, key=DOMAIN_PRIORITY.index) for a, ds in artist_domains.items() if len(ds) > 1}
kept = []
for r in records:
    owner = shared.get(r['artist_key'])
    if owner and r['domain'] != owner:
        exclude(r['dataset'], r['track_id'], 'artist also in another domain', f'artist "{r["artist"]}" kept in {owner}')
    else:
        kept.append(r)
log(f'artist shared across domains: {len(shared)} artists, {len(records) - len(kept)} recordings removed')
CATALOG = sorted(kept, key=lambda r: (DATASET_PRIORITY.index(r['dataset']), priority(r)))
log(f'✅ Step A5 done: {len(CATALOG)} recordings in the clean catalog.')

[09:08:22] identical audio file: 92 recordings removed
[09:08:22] same artist and title: 18 recordings removed
[09:08:22] artist shared across domains: 1 artists, 1 recordings removed
[09:08:22] ✅ Step A5 done: 5063 recordings in the clean catalog.


## Step A6 — Save the catalog and the audit report

Files written to `OUTPUT_DIR/catalog/`:

| File | Content |
|---|---|
| `catalog.csv` | one row per retained recording: dataset, domain, audio path, checksum, duration, artist, title, split group, labels in 0–1 (empty for Soundtracks) |
| `exclusions.csv` | every removed recording with the reason |
| `memo2496_per_second_ratings.npz` | the original per-second Memo2496 ratings, so other label windows can be tested later |
| `soundtracks_lock.json` | checksum of the Soundtracks rating file; the final test checks that it is unchanged |
| `audit_report.json` | counts, label statistics, durations and all protocol settings |

If a catalog from an earlier session exists, the new one must be **identical** (same recordings, same checksums); otherwise the cell stops, so features from different catalogs can never be mixed.

In [15]:
CATALOG_DIR = OUTPUT_DIR / 'catalog'
CATALOG_DIR.mkdir(parents=True, exist_ok=True)
columns = ['uid', 'dataset', 'domain', 'track_id', 'audio_relpath', 'audio_sha256', 'audio_bytes', 'duration_s',
           'artist', 'title', 'group_id', 'valence', 'arousal', 'label_note', 'label_locked']
catalog_df = pd.DataFrame(CATALOG)[columns]
excluded_df = pd.DataFrame(EXCLUDED, columns=['uid', 'dataset', 'track_id', 'reason', 'detail'])

previous = CATALOG_DIR / 'catalog.csv'
if previous.is_file() and (CATALOG_DIR / 'audit_report.json').is_file():
    old = pd.read_csv(previous, dtype=str, keep_default_na=False)
    same = sorted(zip(old['uid'], old['audio_sha256'])) == sorted(zip(catalog_df['uid'], catalog_df['audio_sha256']))
    old_protocol = json.loads((CATALOG_DIR / 'audit_report.json').read_text())['protocol_hash']
    if not same or old_protocol != PROTOCOL_HASH:
        raise RuntimeError('The catalog differs from the one saved earlier (different files or settings). '
                           'Features from both versions must not be mixed. If the change is intended, delete '
                           f'{OUTPUT_DIR} on Drive and run the notebook from the start.')
    log('Catalog identical to the saved one (expected when resuming).')

catalog_df.to_csv(CATALOG_DIR / 'catalog.csv.tmp', index=False)
os.replace(CATALOG_DIR / 'catalog.csv.tmp', CATALOG_DIR / 'catalog.csv')
excluded_df.to_csv(CATALOG_DIR / 'exclusions.csv', index=False)
memo_uids = sorted(MEMO_SERIES)
np.savez_compressed(CATALOG_DIR / 'memo2496_per_second_ratings.npz', uids=np.array(memo_uids),
                    **{f's{i}': MEMO_SERIES[u].astype(np.float32) for i, u in enumerate(memo_uids)})
atomic_json(CATALOG_DIR / 'soundtracks_lock.json', SOUNDTRACKS_LOCK)

def describe(values):
    values = np.asarray(values, dtype=float)
    return dict(n=int(len(values)), mean=round(float(values.mean()), 4), sd=round(float(values.std()), 4),
                min=round(float(values.min()), 4), max=round(float(values.max()), 4))

report = dict(protocol=PROTOCOL, protocol_hash=PROTOCOL_HASH, created=datetime.datetime.now().isoformat(),
              retained=catalog_df.groupby('dataset').size().to_dict(),
              retained_by_domain=catalog_df.groupby('domain').size().to_dict(),
              excluded=excluded_df.groupby(['dataset', 'reason']).size().rename('n').reset_index().to_dict('records'),
              duration_s={d: describe(g['duration_s']) for d, g in catalog_df.groupby('dataset')},
              labels={d: dict(valence=describe(g['valence']), arousal=describe(g['arousal']))
                      for d, g in catalog_df[~catalog_df['label_locked']].groupby('dataset')},
              soundtracks_lock=SOUNDTRACKS_LOCK)
atomic_json(CATALOG_DIR / 'audit_report.json', report)

print('Recordings kept per dataset and domain:')
display(catalog_df.groupby(['domain', 'dataset']).size().rename('recordings').to_frame())
print('Removed recordings by reason:')
display(excluded_df.groupby(['dataset', 'reason']).size().rename('recordings').to_frame())
print('Duration in seconds:')
display(pd.DataFrame(report['duration_s']).T)
print('Labels on the 0–1 scale (Soundtracks locked, not shown):')
display(pd.DataFrame({(d, ax): s for d, axes in report['labels'].items() for ax, s in axes.items()}).T)
log(f'✅ Part A finished. Catalog saved to {CATALOG_DIR}')

Recordings kept per dataset and domain:


recordings
domain      dataset                
MediaEval   DEAM               1044
            EmoMusic            744
Memo2496    Memo2496           2414
PMEmo       PMEmo               754
Soundtracks Soundtracks         107

Removed recordings by reason:


recordings
dataset     reason                                                        
DEAM        DEAM 2013 subset (identical to the EmoMusic cat...         744
            artist also in another domain                                1
            same artist and title                                       13
EmoMusic    audio without rating                                       256
Memo2496    identical audio file                                        82
PMEmo       audio without rating                                        27
            identical audio file                                         8
            same artist and title                                        5
Soundtracks duplicate excerpt                                            1
            identical audio file                                         2

Duration in seconds:


,n,mean,sd,min,max
DEAM,1044.0,55.5825,50.0534,44.860,628.616
EmoMusic,744.0,45.0389,0.0207,44.631,45.061
Memo2496,2414.0,156.3094,49.0486,16.057,300.425
PMEmo,754.0,38.8732,13.6478,16.082,88.242
Soundtracks,107.0,15.3225,1.9547,11.073,27.048


Labels on the 0–1 scale (Soundtracks locked, not shown):


n    mean      sd     min     max
DEAM     valence  1044.0  0.4976  0.1250  0.1500  0.8625
         arousal  1044.0  0.4585  0.1607  0.1000  0.8500
EmoMusic valence   744.0  0.5008  0.1556  0.0750  0.8875
         arousal   744.0  0.4733  0.1719  0.0750  0.9250
Memo2496 valence  2414.0  0.5406  0.1137  0.2338  0.8372
         arousal  2414.0  0.4913  0.1620  0.1341  0.8962
PMEmo    valence   754.0  0.5960  0.1620  0.1250  0.9125
         arousal   754.0  0.6213  0.1839  0.0875  0.9750

[09:08:24] ✅ Part A finished. Catalog saved to /content/drive/MyDrive/Thesis - Music Emotion Recognition/MER_DG_v2/catalog


# Part B — All-layer MERT feature extraction

## Step B1 — Load the MERT model on the GPU

The exact model version (a fixed Hugging Face revision code) is recorded the first time and reused in every later session, so all features come from the same model.
The model is used **frozen**: it is never trained, and it never sees any rating.

In [16]:
import torch
from transformers import AutoModel, Wav2Vec2FeatureExtractor
from huggingface_hub import model_info

if not torch.cuda.is_available():
    raise RuntimeError('No GPU found. Runtime → Change runtime type → choose a GPU, then Run all.')
DEVICE = torch.device('cuda')
FEATURE_DIR = OUTPUT_DIR / 'features'
FEATURE_DIR.mkdir(parents=True, exist_ok=True)
extractor_file = FEATURE_DIR / 'extractor.json'
if extractor_file.is_file():
    REVISION = json.loads(extractor_file.read_text())['revision']
    log(f'Reusing the recorded MERT revision {REVISION[:10]}')
else:
    REVISION = model_info(MODEL_ID).sha
    log(f'Recording MERT revision {REVISION[:10]} for all future sessions')

processor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_ID, revision=REVISION, trust_remote_code=True)
model = AutoModel.from_pretrained(MODEL_ID, revision=REVISION, trust_remote_code=True).to(DEVICE).eval()
if processor.sampling_rate != SAMPLE_RATE:
    raise ValueError(f'MERT expects {processor.sampling_rate} Hz, settings say {SAMPLE_RATE} Hz')
N_LAYERS = model.config.num_hidden_layers + 1   # embedding output + every transformer layer
WIDTH = model.config.hidden_size
FEATURE_SETTINGS = dict(model_id=MODEL_ID, revision=REVISION, sample_rate=SAMPLE_RATE, window_s=WINDOW_S,
                        min_last_window_s=MIN_LAST_WINDOW_S, n_layers=N_LAYERS, width=WIDTH,
                        pooling='mean over frames within each window, every hidden state',
                        decoding='soundfile (ffmpeg fallback), channel mean, scipy resample_poly',
                        input_normalization='MERT feature extractor defaults',
                        catalog_sha256=sha256_file(CATALOG_DIR / 'catalog.csv'), protocol_hash=PROTOCOL_HASH)
SETTINGS_HASH = sha256_json(FEATURE_SETTINGS)
if extractor_file.is_file() and json.loads(extractor_file.read_text())['settings_hash'] != SETTINGS_HASH:
    raise RuntimeError('Feature settings differ from those used for the features already on Drive. '
                       f'Do not mix them: delete {FEATURE_DIR} to start the extraction again.')
atomic_json(extractor_file, dict(FEATURE_SETTINGS, settings_hash=SETTINGS_HASH,
                                 processor_config=processor.to_dict(), torch=torch.__version__,
                                 gpu=torch.cuda.get_device_name(0)))
log(f'✅ MERT loaded on {torch.cuda.get_device_name(0)}: {N_LAYERS} layers × {WIDTH} values per window.')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


[09:08:35] Recording MERT revision 12af15fef9 for all future sessions


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/211 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

configuration_MERT.py: 0.00B [00:00, ?B/s]

modeling_MERT.py: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Some weights of the model checkpoint at m-a-p/MERT-v1-95M were not used when initializing MERTModel: ['encoder.pos_conv_embed.conv.weight_g', 'encoder.pos_conv_embed.conv.weight_v']
- This IS expected if you are initializing MERTModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing MERTModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of MERTModel were not initialized from the model checkpoint at m-a-p/MERT-v1-95M and are newly initialized: ['encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'encoder.pos_conv_embed.conv.parametrizations.weight.original1']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[09:08:42] ✅ MERT loaded on NVIDIA L4: 13 layers × 768 values per window.


## Step B2 — Functions that turn one recording into features

1. **Decode** the audio file, average the channels to mono and resample to 24 kHz.
2. **Cut** it into consecutive 5-second windows (a final piece shorter than 1 s is joined to the previous window). The start and end time of every window is stored, so later we can pick, e.g., only 15–75 s of a Memo2496 track.
3. **Run MERT** on each window and keep, for **each of the 13 layers**, the average of its outputs over the window.

Result per recording: an array of shape *(number of windows × 13 layers × 768)*, saved as one compressed file.
A quick self-test on a synthetic tone runs at the end of this cell.

In [17]:
from scipy.signal import resample_poly
from concurrent.futures import ThreadPoolExecutor

def load_audio(path):
    path = str(path)
    try:
        data, rate = sf.read(path, dtype='float32', always_2d=True)
        mono = data.mean(axis=1)
    except Exception:
        raw = subprocess.run(['ffmpeg', '-v', 'error', '-i', path, '-f', 'f32le', '-ac', '1', '-ar', str(SAMPLE_RATE), '-'],
                             capture_output=True, check=True).stdout
        mono, rate = np.frombuffer(raw, dtype=np.float32).copy(), SAMPLE_RATE
    if rate != SAMPLE_RATE:
        g = math.gcd(int(rate), SAMPLE_RATE)
        mono = resample_poly(mono, SAMPLE_RATE // g, int(rate) // g).astype(np.float32)
    if not len(mono) or not np.isfinite(mono).all():
        raise ValueError('empty or non-finite audio')
    return mono

def make_windows(n_samples):
    step = int(round(WINDOW_S * SAMPLE_RATE))
    windows = [(s, min(s + step, n_samples)) for s in range(0, n_samples, step)]
    if len(windows) > 1 and windows[-1][1] - windows[-1][0] < MIN_LAST_WINDOW_S * SAMPLE_RATE:
        windows[-2] = (windows[-2][0], n_samples)
        windows.pop()
    return windows

@torch.inference_mode()
def extract(audio):
    windows = make_windows(len(audio))
    layers = np.zeros((len(windows), N_LAYERS, WIDTH), dtype=np.float32)
    frames = np.zeros(len(windows), dtype=np.int32)
    by_length = {}
    for i, (a, b) in enumerate(windows):
        by_length.setdefault(b - a, []).append(i)
    for idx in by_length.values():               # equal-length windows are batched together
        for k in range(0, len(idx), BATCH_WINDOWS):
            chunk = idx[k:k + BATCH_WINDOWS]
            inputs = processor([audio[windows[i][0]:windows[i][1]] for i in chunk],
                               sampling_rate=SAMPLE_RATE, return_tensors='pt')
            hidden = model(input_values=inputs['input_values'].to(DEVICE), output_hidden_states=True).hidden_states
            stacked = torch.stack(hidden, dim=1)           # batch × layers × frames × width
            layers[chunk] = stacked.mean(dim=2).float().cpu().numpy()
            frames[chunk] = stacked.shape[2]
    starts = np.array([a / SAMPLE_RATE for a, _ in windows], dtype=np.float32)
    ends = np.array([b / SAMPLE_RATE for _, b in windows], dtype=np.float32)
    return layers, starts, ends, frames

def feature_path(rec):
    safe = re.sub(r'[^A-Za-z0-9_.-]', '_', rec['track_id'])
    return FEATURE_DIR / rec['dataset'] / f'{safe}.npz'

def expected_meta(rec):
    return dict(uid=rec['uid'], audio_sha256=rec['audio_sha256'], settings_hash=SETTINGS_HASH)

def save_features(rec, layers, starts, ends, frames):
    path = feature_path(rec)
    path.parent.mkdir(parents=True, exist_ok=True)
    stored = layers.astype(np.float16) if np.abs(layers).max() < 60000 else layers   # float16 halves disk use
    tmp = path.with_name(path.name + '.tmp.npz')
    np.savez_compressed(tmp, layers=stored, start_s=starts, end_s=ends, frames=frames,
                        meta=np.array(json.dumps(expected_meta(rec), sort_keys=True)))
    os.replace(tmp, path)

def features_valid(rec):
    path = feature_path(rec)
    if not path.is_file():
        return False
    try:
        with np.load(path, allow_pickle=False) as f:
            meta = json.loads(str(f['meta']))
            layers = f['layers']
            ok = (meta == expected_meta(rec) and layers.ndim == 3 and layers.shape[1:] == (N_LAYERS, WIDTH)
                  and len(f['start_s']) == len(layers) == len(f['end_s']) == len(f['frames'])
                  and np.isfinite(layers).all())
        return bool(ok)
    except Exception:
        return False

# Self-test on a synthetic 12.3-second tone: 2 full windows + 2.3 s remainder kept as a third window.
tone = (0.1 * np.sin(2 * np.pi * 440 * np.arange(int(12.3 * SAMPLE_RATE)) / SAMPLE_RATE)).astype(np.float32)
assert make_windows(len(tone)) == [(0, 120000), (120000, 240000), (240000, 295200)]
assert make_windows(int(10.5 * SAMPLE_RATE)) == [(0, 120000), (120000, 252000)]   # 0.5 s remainder merged
test_layers, test_starts, test_ends, test_frames = extract(tone)
assert test_layers.shape == (3, N_LAYERS, WIDTH) and np.isfinite(test_layers).all()
assert np.allclose(test_starts, [0, 5, 10]) and np.allclose(test_ends, [5, 10, 12.3], atol=1e-3)
log(f'✅ Self-test passed: synthetic tone → {test_layers.shape} features, {list(test_frames)} frames per window.')

[09:08:48] ✅ Self-test passed: synthetic tone → (3, 13, 768) features, [374, 374, 172] frames per window.


## Step B3 — Extract features for every recording (the long step)

* Datasets are processed one after another; for each one the cell prints how many recordings are done, the elapsed time and the **remaining time**, at least once per minute.
* While the GPU works on one recording, the next ones are decoded in parallel.
* Each recording is saved to Drive as soon as it is finished. **If Colab disconnects, just Run all again**: recordings already saved are checked and skipped.
* Recordings that fail are listed at the end (and written to `extraction_failures.csv`) instead of stopping the whole run.

In [18]:
EXTRACTION_ORDER = ['Soundtracks', 'PMEmo', 'EmoMusic', 'DEAM', 'Memo2496']
failures = []
run_start = time.monotonic()
for dataset in EXTRACTION_ORDER:
    items = [r for r in CATALOG if r['dataset'] == dataset]
    todo = [r for r in items if not features_valid(r)]
    log(f'━━━ {dataset}: {len(items)} recordings, {len(items) - len(todo)} already done, {len(todo)} to extract ━━━')
    if not todo:
        continue
    progress = Progress(len(todo), f'{dataset} feature extraction', every_s=60)
    with ThreadPoolExecutor(max_workers=4) as pool:
        for k in range(0, len(todo), PREFETCH_TRACKS):
            chunk = todo[k:k + PREFETCH_TRACKS]
            futures = [pool.submit(load_audio, LOCAL_RAW / r['audio_relpath']) for r in chunk]
            for rec, future in zip(chunk, futures):
                try:
                    audio = future.result()
                    if abs(len(audio) / SAMPLE_RATE - rec['duration_s']) > 2.0:
                        log(f'  ⚠ {rec["uid"]}: decoded length {len(audio) / SAMPLE_RATE:.1f} s vs catalog {rec["duration_s"]:.1f} s')
                    layers, starts, ends, frames = extract(audio)
                    save_features(rec, layers, starts, ends, frames)
                    note = f'| last: {rec["uid"]} ({len(starts)} windows)'
                except Exception as error:
                    failures.append(dict(uid=rec['uid'], error=f'{type(error).__name__}: {error}'[:300]))
                    log(f'  ❌ {rec["uid"]} failed: {failures[-1]["error"]}')
                    note = '| last item failed'
                progress.step(note=note)
    progress.finish()
pd.DataFrame(failures, columns=['uid', 'error']).to_csv(FEATURE_DIR / 'extraction_failures.csv', index=False)
log(f'✅ Step B3 finished in {fmt_seconds(time.monotonic() - run_start)}; {len(failures)} failures.')

[09:08:48] ━━━ Soundtracks: 107 recordings, 0 already done, 107 to extract ━━━
[09:08:48] ▶ Soundtracks feature extraction: starting (107 items)
[09:08:49]   Soundtracks feature extraction: 1/107 (0.9%) | elapsed 0m 00s | remaining ≈ 0m 14s | last: Soundtracks:001 (3 windows)
[09:08:56]   Soundtracks feature extraction: 107/107 (100.0%) | elapsed 0m 07s | remaining ≈ 0m 00s | last: Soundtracks:110 (4 windows)
[09:08:56] ✔ Soundtracks feature extraction: finished in 0m 07s
[09:08:56] ━━━ PMEmo: 754 recordings, 0 already done, 754 to extract ━━━
[09:08:56] ▶ PMEmo feature extraction: starting (754 items)
[09:08:56]   PMEmo feature extraction: 1/754 (0.1%) | elapsed 0m 00s | remaining ≈ 2m 58s | last: PMEmo:1 (7 windows)
[09:09:57]   PMEmo feature extraction: 444/754 (58.9%) | elapsed 1m 00s | remaining ≈ 0m 42s | last: PMEmo:582 (13 windows)
[09:10:43]   PMEmo feature extraction: 754/754 (100.0%) | elapsed 1m 46s | remaining ≈ 0m 00s | last: PMEmo:1000 (16 windows)
[09:10:43] ✔ PMEmo fea

## Step B4 — Verify all features and write the feature manifest

Every saved file is re-opened and checked (right recording, right audio checksum, right settings, right shape, no invalid numbers).
The manifest `features/manifest.csv` lists each recording with its number of windows; the Training Three Folds notebook trains only on recordings listed there.

In [19]:
rows, missing = [], []
progress = Progress(len(CATALOG), 'Verifying feature files', every_s=30)
for rec in CATALOG:
    if features_valid(rec):
        path = feature_path(rec)
        with np.load(path, allow_pickle=False) as f:
            n_windows, last_end = len(f['start_s']), float(f['end_s'][-1])
        rows.append(dict(uid=rec['uid'], dataset=rec['dataset'], domain=rec['domain'],
                         feature_relpath=str(path.relative_to(OUTPUT_DIR)), n_windows=n_windows,
                         covered_s=round(last_end, 2), file_bytes=path.stat().st_size))
    else:
        missing.append(rec['uid'])
    progress.step()
progress.finish()
manifest = pd.DataFrame(rows)
manifest.to_csv(FEATURE_DIR / 'manifest.csv', index=False)
overview = manifest.groupby('dataset').agg(recordings=('uid', 'size'), windows=('n_windows', 'sum'),
                                           size_MB=('file_bytes', lambda b: round(b.sum() / 2**20, 1)))
display(overview)
if missing:
    print(f'⚠ {len(missing)} recordings have no valid features yet, e.g. {missing[:5]}.')
    print('  Run all again to retry them. If they keep failing, see extraction_failures.csv.')
else:
    log('✅ Every catalog recording has verified features.')

[09:39:00] ▶ Verifying feature files: starting (5063 items)
[09:39:00]   Verifying feature files: 1/5063 (0.0%) | elapsed 0m 00s | remaining ≈ 0m 39s 
[09:39:30]   Verifying feature files: 3468/5063 (68.5%) | elapsed 0m 30s | remaining ≈ 0m 13s 
[09:39:49]   Verifying feature files: 5063/5063 (100.0%) | elapsed 0m 48s | remaining ≈ 0m 00s 
[09:39:49] ✔ Verifying feature files: finished in 0m 48s


,recordings,windows,size_MB
dataset,,,
DEAM,1044,11607,206.7
EmoMusic,744,6696,119.4
Memo2496,2414,76112,1350.2
PMEmo,754,6146,109.6
Soundtracks,107,344,6.2


[09:39:49] ✅ Every catalog recording has verified features.


## Step B5 — Report possible near-duplicate recordings across datasets

Byte checksums only find *identical files*. The same song encoded differently is found here by comparing the average MERT representation (layer 6) of every pair of recordings.
Pairs with a cosine similarity of 0.98 or more are written to `near_duplicates.csv` **for manual checking only**: nothing is removed automatically. Ratings are not used.

In [20]:
MID_LAYER = 6
vectors, uids, domains = [], [], []
for row in manifest.itertuples():
    with np.load(OUTPUT_DIR / row.feature_relpath, allow_pickle=False) as f:
        vectors.append(f['layers'][:, MID_LAYER, :].astype(np.float32).mean(axis=0))
    uids.append(row.uid)
    domains.append(row.domain)
X = np.stack(vectors)
X /= np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-8)
similarity = X @ X.T
i, j = np.where(np.triu(similarity >= 0.98, k=1))
domains = np.array(domains)
near = pd.DataFrame(dict(uid_a=np.array(uids)[i], uid_b=np.array(uids)[j], cosine=similarity[i, j].round(5),
                         same_domain=domains[i] == domains[j])).sort_values('cosine', ascending=False)
near.to_csv(FEATURE_DIR / 'near_duplicates.csv', index=False)
print(f'Pairs with cosine ≥ 0.98: {len(near)} (across domains: {int((~near.same_domain).sum())})')
display(near.head(20))

Pairs with cosine ≥ 0.98: 117 (across domains: 0)


,uid_a,uid_b,cosine,same_domain
0,Soundtracks:022,Soundtracks:075,0.99990,True
2,Soundtracks:032,Soundtracks:086,0.99974,True
37,Memo2496:294,Memo2496:372,0.99973,True
15,DEAM:1156,DEAM:1160,0.99953,True
1,Soundtracks:024,Soundtracks:072,0.99950,True
28,Memo2496:123,Memo2496:142,0.99943,True
17,DEAM:1158,DEAM:1159,0.99897,True
77,Memo2496:1305,Memo2496:2015,0.99869,True
54,Memo2496:544,Memo2496:684,0.99837,True
61,Memo2496:664,Memo2496:1917,0.99805,True


## Step B6 — Create the review file to send back

Packs the small result files (catalog, exclusions, audit report, feature manifest, near-duplicate report, failures, extractor settings) into `data_preparation_review.zip` in `OUTPUT_DIR`. Feature arrays and audio are **not** included.
Download it from Drive (or with the link below) and send it.

In [21]:
review = OUTPUT_DIR / 'data_preparation_review.zip'
with zipfile.ZipFile(review, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in list(CATALOG_DIR.glob('*')) + [FEATURE_DIR / n for n in
                 ('manifest.csv', 'near_duplicates.csv', 'extraction_failures.csv', 'extractor.json')]:
        if path.is_file() and path.suffix in ('.csv', '.json'):
            z.write(path, path.relative_to(OUTPUT_DIR).as_posix())
log(f'✅ Review file ready on Drive: {review}')
try:
    from google.colab import files
    files.download(str(review))   # also starts a browser download
except Exception as error:
    print('Automatic download not available; take the file from Drive instead.', error)

[09:40:18] ✅ Review file ready on Drive: /content/drive/MyDrive/Thesis - Music Emotion Recognition/MER_DG_v2/data_preparation_review.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>